In [2]:
import pandas as pd

In [5]:
df = pd.read_csv('Данные.csv')

df['date'] = pd.to_datetime(df['date'])

df.head()

,date,user_id,view_adverts
0,2023-11-11,8c020470-8461-11ed-83d0-552e8cc749d6,13
1,2023-11-18,5875f070-7b92-11ee-a6fb-8b298e83f4f7,14
2,2023-11-29,3c2d27c0-4fd6-11eb-b89f-2ffb31b67dd6,21
3,2023-11-29,234a96d0-ad16-11ed-a2e6-793ddfeeba1f,23
4,2023-11-29,4d07c180-644f-11eb-879c-b7c02edf4f37,12


In [6]:
mau_report = df.groupby(df['date'].dt.to_period('M'))['user_id'].nunique()

print("MAU по месяцам:")
print(mau_report)

MAU по месяцам:
date
2023-11    7639
Freq: M, Name: user_id, dtype: int64


In [9]:
dau_series = df.groupby(df['date'].dt.date)['user_id'].nunique()
avg_dau = dau_series.mean()

print(avg_dau)

560.4666666666667


In [12]:
first_visit = df.groupby('user_id')['date'].min()

users = first_visit[first_visit == '2023-11-01'].index
size = len(users)

d1_active_users = df[
    (df['date'] == '2023-11-02') &
    (df['user_id'].isin(users))
]['user_id'].nunique()

print(f"Retention 1-го дня: {retention_d1:.1f}%")

Retention 1-го дня: 26.6%


In [14]:
total_users = df['user_id'].nunique()
converted_users = df[df['view_adverts'] > 0]['user_id'].nunique()
conversion_rate = (converted_users / total_users) * 100

print(f"Всего уникальных пользователей: {total_users}")
print(f"Из них посмотрели рекламу: {converted_users}")
print(f"Пользовательская конверсия в просмотр: {conversion_rate:.1f}%")

Всего уникальных пользователей: 7639
Из них посмотрели рекламу: 3538
Пользовательская конверсия в просмотр: 46.3%


In [16]:
user_totals = df.groupby('user_id')['view_adverts'].sum()
avg_views = user_totals.mean()

print(f"Среднее количество просмотров на пользователя: {avg_views:.2f}")

Среднее количество просмотров на пользователя: 2.87


In [17]:
total = 2000
promoters = 1200
detractors = 500
passives = 300

pct_promoters = (promoters / total) * 100
pct_detractors = (detractors / total) * 100

nps = pct_promoters - pct_detractors

print(f"Процент сторонников: {pct_promoters:.0f}%")
print(f"Процент критиков: {pct_detractors:.0f}%")
print(f"NPS = {nps:.0f}")

Процент сторонников: 60%
Процент критиков: 25%
NPS = 35


In [19]:
ab = pd.read_csv('Данные АБ тестов.csv')

ab.head()

,experiment_num,experiment_group,user_id,revenue
0,1,test,38456,520
1,1,control,13125924,806
2,1,control,9761984,0
3,1,test,11387012,208
4,1,test,18319648,104


In [20]:
from scipy import stats

In [22]:
results = []

# Проходимся циклом по каждому эксперименту (1, 2, 3)
for exp in ab['experiment_num'].unique():
    exp_data = ab[ab['experiment_num'] == exp]

    # Разделяем на группы (обычно 'control' и 'test' или 'A' и 'B')
    control = exp_data[exp_data['experiment_group'] == 'control']
    test = exp_data[exp_data['experiment_group'] == 'test']

    # 1. Считаем ARPU (общая выручка / количество пользователей)
    arpu_control = control['revenue'].sum() / control['user_id'].nunique()
    arpu_test = test['revenue'].sum() / test['user_id'].nunique()

    # 2. Считаем прирост (Uplift)
    uplift = (arpu_test - arpu_control) / arpu_control * 100

    # 3. Считаем p-value (T-тест Уэлча)
    # Важно: передаем массивы выручки всех пользователей, включая нули
    t_stat, p_value = stats.ttest_ind(test['revenue'], control['revenue'], equal_var=False)

    results.append({
        'Эксперимент': exp,
        'ARPU Control': round(arpu_control, 2),
        'ARPU Test': round(arpu_test, 2),
        'Uplift, %': round(uplift, 1),
        'p-value': round(p_value, 4)
    })

# Выводим красивую табличку
results_df = pd.DataFrame(results)
print(results_df)

   Эксперимент  ARPU Control  ARPU Test  Uplift, %  p-value
0            1        722.46     665.74       -7.9   0.6890
1            2        704.65     332.93      -52.8   0.0011
2            3        663.21     998.67       50.6   0.0603


In [23]:
listers = pd.read_csv('Листеры.csv')

listers.head()

,user_id,date,cnt_adverts,age,cnt_contacts,revenue
0,100,2022-01-01,6,21,119,53
1,100,2022-01-02,2,21,200,18
2,100,2022-01-03,6,21,193,42
3,100,2022-01-04,2,21,143,38
4,100,2022-01-05,2,21,190,40


In [28]:
unique_listers = listers.drop_duplicates(subset=['user_id'])
total_revenue = listers['revenue'].sum()
arpu = total_revenue / unique_users

print(f"ARPU: {arpu:.2f}")

ARPU: 156.48


In [29]:
median_age = unique_listers['age'].median()

print(f"Медиана возраста пользователя: {median_age}")

Медиана возраста пользователя: 28.0


In [32]:
import numpy as np
from statsmodels.stats.proportion import proportions_ztest

# Вариант A
visitors_a = 100_047_501
conversions_a = 1_003

# Вариант B
visitors_b = 100_001_055
conversions_b = 1_099

cr_a = conversions_a / visitors_a
cr_b = conversions_b / visitors_b

print("Конверсия:")
print(f"Вариант A (Control): {cr_a * 100:.5f}% ({cr_a * 100_000:.2f} платежей на 100к юзеров)")
print(f"Вариант B (Test):    {cr_b * 100:.5f}% ({cr_b * 100_000:.2f} платежей на 100к юзеров)")

uplift = (cr_b - cr_a) / cr_a * 100
print(f"\nОтносительный прирост (Uplift): +{uplift:.2f}%")

count = np.array([conversions_b, conversions_a])
nobs = np.array([visitors_b, visitors_a])

z_stat, p_value = proportions_ztest(count, nobs, alternative='two-sided')

print(f"Z-статистика: {z_stat:.3f}")
print(f"p-value: {p_value:.3f}")

alpha = 0.05
if p_value < alpha:
    print("Результат статистически значим (p-value < 0.05).")
    if uplift > 0:
        print("Рекомендация: Тестовая группа B лучше. Можно внедрять (Rollout).")
    else:
        print("Рекомендация: Тестовая группа B хуже. Откатить изменения (Rollback).")
else:
    print("Результат НЕ является статистически значимым.")
    print("Рекомендация: Разница в пределах погрешности. Внедрять нет смысла.")

Конверсия:
Вариант A (Control): 0.00100% (1.00 платежей на 100к юзеров)
Вариант B (Test):    0.00110% (1.10 платежей на 100к юзеров)

Относительный прирост (Uplift): +9.62%
Z-статистика: 2.105
p-value: 0.035
Результат статистически значим (p-value < 0.05).
Рекомендация: Тестовая группа B лучше. Можно внедрять (Rollout).
